# SI — absolute errors behind the Fig. 1c fit yield

**Makes up:** the SI figure that accompanies Fig. 1c. The Fig. 1c yield counts a fit as successful when it is
accepted (SI, rejected fits) and its x_c, y_c and each relative pixel QE lie within 3σ_err of the ground truth,
with σ_err each fit's *own* uncertainty (covariance at the minimum). That tolerance scales with the fit's
uncertainty, so a high yield on its own does not mean small errors. This notebook reports the absolute errors
and where the failures come from, from the same fits.

**Data:** the Fig. 1c sweep itself (`Figure_1c.ipynb`, full mode): ATTO 488, ATTO 565 and ATTO 647N emitting
1,000 photons, 100 read-noise values (0.1–10 RMS e⁻, log-spaced) × 60 peak pixel QEs (0–1), 20,000 simulated
molecules per point, each fitted in a box placed around its known position (no detection step). Every fit was
saved (`*_rawresults.h5`, rejected fits as NaN rows); nothing is simulated again here.

**Metrics per grid point** (denominator: all 20,000 simulated molecules):
- *rejected*: fraction of fits rejected (no χ² improvement, outside the box, or failing the Δχ² emitter test);
- *outside 3σ_err*: accepted, but x_c, y_c or a relative pixel QE more than 3σ_err from the truth;
- *yield*: the rest (as Fig. 1c);
- *position error*: radial distance of the fitted from the true position (nm);
- *fingerprint error*: Euclidean distance of the fitted (A_B, A_G, A_R) from the dye's true fingerprint
  (relative pixel QE).

The error maps show the median over all accepted fits, not only those counted as successful: filtering by the
σ_err-scaled test first would hide fits with a large error but a small σ_err. The medians over the successful fits
are in the summary table too. White dots are the cameras of Fig. 1c.

---
## 0 — Imports

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyS3M import PlottingBase

plotter = PlottingBase.PublicationPlotter(dark_background=False)

---
## 1 — Configuration

In [ ]:
REPO_ROOT = Path.cwd().resolve().parents[2]   # notebooks/figures/SI -> repo root
# The full Fig. 1c sweep, as Figure_1c.ipynb writes it with FAST_MODE = False
SWEEP_DIR = REPO_ROOT / "notebooks" / "figures" / "outputs" / "Figure_1c" / "full"
if not SWEEP_DIR.is_dir():
    raise FileNotFoundError(f"{SWEEP_DIR} not found: run notebooks/figures/Figure_1c.ipynb with FAST_MODE = False first")
OUT_ROOT = REPO_ROOT / "notebooks" / "figures" / "outputs" / "FigureSI_Fig1c_ErrorMetrics"   # git-ignored
FIG_DIR = OUT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = OUT_ROOT / "error_metrics_by_gridpoint.csv"
OVERWRITE = False      # True: recompute the summary from the raw fits even if it exists
N_WORKERS = 8

# The Fig. 1c sweep (Figure_1c.ipynb, FAST_MODE = False)
DYES = ["ATTO 488", "ATTO 565", "ATTO 647N"]
N_PHOTONS = 1000
READ_NOISE = np.logspace(np.log10(0.1), np.log10(10.0), 100)
PEAK_QE = np.linspace(0.0, 1.0, 60)
PIXEL_SIZE_NM = 69.0
POS_ERR_MULTIPLIER = 3.0       # as Figure_1c.ipynb
COLOUR_ERR_MULTIPLIER = 3.0
YIELD_BENCHMARK = 0.8          # the 80% benchmark of Fig. 1c

camera_data = pd.read_excel(REPO_ROOT / "notebooks" / "figures" / "Camera_History.ods")
camera_data = camera_data[camera_data["Year"] >= 2018].reset_index(drop=True)
camera_data[["Model", "Year", "RMS", "PeakQY", "Company"]]

---
## 2 — Metrics from the raw fits

One pass over the 18,000 result files (3 dyes × 6,000 grid points), in parallel; the per-point summary is cached
in `SUMMARY_CSV`. The yield column reproduces Fig. 1c's criterion exactly (`Figure_1c.ipynb`, cell "recovery").

In [ ]:
def grid_point_metrics(args):
    dye, i, j, true_colour = args
    flag = f"{dye.replace(' ', '-')}_qy_{PEAK_QE[i]:.3f}_readnoise_{READ_NOISE[j]:.6f}_LM_method_{dye}"
    raw, gt_path = SWEEP_DIR / f"{flag}_rawresults.h5", SWEEP_DIR / f"{flag}_fittesting_input_groundtruthpositions.csv"
    row = dict(dye=dye, i_qe=i, j_rn=j, peak_qe=PEAK_QE[i], read_noise=READ_NOISE[j])
    if not (raw.exists() and gt_path.exists()):
        return row
    r = pd.read_hdf(raw)
    r = r[r["photon_level"] == 0]
    gt = pd.read_csv(gt_path)
    n = len(r)
    ex = r["xc"].to_numpy() - gt["x0"].to_numpy() / PIXEL_SIZE_NM
    ey = r["yc"].to_numpy() - gt["y0"].to_numpy() / PIXEL_SIZE_NM
    accepted = np.isfinite(ex) & np.isfinite(ey)
    with np.errstate(invalid="ignore"):
        pos_ok = accepted & (np.abs(ex) < POS_ERR_MULTIPLIER * r["xc_err"].to_numpy()) \
                          & (np.abs(ey) < POS_ERR_MULTIPLIER * r["yc_err"].to_numpy())
    fit_fp = r[["A_B", "A_G", "A_R"]].to_numpy()
    d_fp = fit_fp - true_colour
    with np.errstate(invalid="ignore"):   # rejected fits are NaN rows: they fail every criterion
        colour_ok = np.all(np.abs(d_fp) < COLOUR_ERR_MULTIPLIER * r[["A_B_err", "A_G_err", "A_R_err"]].to_numpy(), axis=1)
    success = pos_ok & colour_ok
    pos_nm = np.hypot(ex, ey) * PIXEL_SIZE_NM
    fp_err = np.linalg.norm(d_fp, axis=1)
    sig_nm = np.sqrt((r["xc_err"].to_numpy() ** 2 + r["yc_err"].to_numpy() ** 2) / 2) * PIXEL_SIZE_NM

    def med(v, m):
        return float(np.median(v[m])) if m.any() else np.nan

    def p95(v, m):
        return float(np.percentile(v[m], 95)) if m.any() else np.nan

    row.update(
        n=n, frac_rejected=1 - accepted.mean(), frac_outside_3sigma=(accepted & ~success).mean(),
        frac_outside_pos=(accepted & ~pos_ok).mean(), frac_outside_colour=(accepted & ~colour_ok).mean(),
        yield_=success.mean(),
        pos_err_nm_median_success=med(pos_nm, success), pos_err_nm_p95_success=p95(pos_nm, success),
        fp_err_median_success=med(fp_err, success), fp_err_p95_success=p95(fp_err, success),
        pos_err_nm_median_accepted=med(pos_nm, accepted), pos_err_nm_p95_accepted=p95(pos_nm, accepted),
        fp_err_median_accepted=med(fp_err, accepted), fp_err_p95_accepted=p95(fp_err, accepted),
        sigma_err_pos_nm_median_accepted=med(sig_nm, accepted))
    return row


if SUMMARY_CSV.exists() and not OVERWRITE:
    summary = pd.read_csv(SUMMARY_CSV)
else:
    jobs = []
    for dye in DYES:
        tc_path = SWEEP_DIR / f"{dye}_nphot_{N_PHOTONS}_true_colour_fractions.csv"
        tc = pd.read_csv(tc_path)[["true_A_B", "true_A_G", "true_A_R"]].to_numpy()[0]   # every row is the same
        jobs += [(dye, i, j, tc) for i in range(len(PEAK_QE)) for j in range(len(READ_NOISE))]
    with ProcessPoolExecutor(N_WORKERS) as ex:
        summary = pd.DataFrame(list(ex.map(grid_point_metrics, jobs, chunksize=50)))
    if summary["n"].notna().all():
        summary.to_csv(SUMMARY_CSV, index=False)
    else:
        print(f"{summary['n'].isna().sum()} grid points have no results yet: summary not cached")
print(summary.groupby("dye")["n"].agg(["count", "min", "max"]))
summary.groupby("dye")[["yield_", "frac_rejected", "frac_outside_3sigma"]].agg(["min", "max"]).round(3)

---
## 3 — Figure

Rows: a) median position error and b) median fingerprint error of all accepted fits;
c) fraction of all simulated molecules whose fit was rejected; d) fraction accepted but outside 3σ_err.
Columns: dyes. White dots: the cameras of Fig. 1c. 100 filled levels per surface, so the range reads as continuous.

In [ ]:
from matplotlib.ticker import MaxNLocator


def surface(dye, col):
    s = summary[summary["dye"] == dye]
    z = np.full((len(PEAK_QE), len(READ_NOISE)), np.nan)
    z[s["i_qe"].to_numpy(), s["j_rn"].to_numpy()] = s[col].to_numpy()
    return z


ROWS = [
    ("pos_err_nm_median_accepted", "median position error (nm)", 1),
    ("fp_err_median_accepted", "median fingerprint error\n(relative pixel QE)", 1),
    ("frac_rejected", "rejected (%)", 100),
    ("frac_outside_3sigma", "accepted, outside 3σ$_\\mathrm{err}$ (%)", 100),
]
N_LEVELS = 100

fig, axs = plotter.two_column_plot(nrows=len(ROWS), ncols=len(DYES), width=6.75, height=2.1 * len(ROWS))
axs = np.atleast_2d(axs)
for r, (col, label, scale) in enumerate(ROWS):
    zs = [surface(d, col) * scale for d in DYES]
    finite = np.concatenate([z[np.isfinite(z)] for z in zs])
    vmax = np.nanpercentile(finite, 99) if len(finite) else 1.0
    levels = np.linspace(0, vmax, N_LEVELS + 1)
    for c, dye in enumerate(DYES):
        ax = axs[r, c]
        im = ax.contourf(READ_NOISE, PEAK_QE, np.clip(zs[c], 0, vmax), levels=levels, cmap="viridis", extend="neither",
                         antialiased=False)
        im.set_rasterized(True)   # 100 bands: raster (600 dpi) avoids seams between them and keeps the file small
        ax.scatter(camera_data["RMS"], camera_data["PeakQY"], s=10, lw=0.5, facecolor="white", edgecolors="k", zorder=5)
        ax.set_xscale("log")
        ax.set_xlim(READ_NOISE[0], READ_NOISE[-1])
        ax.set_ylim(0, 1)
        ax.grid(False)
        if r == len(ROWS) - 1:
            ax.set_xlabel("read noise (RMS e$^-$)")
        else:
            ax.set_xticklabels([])
        if c == 0:
            ax.set_ylabel("peak pixel QE")
        else:
            ax.set_yticklabels([])
        if r == 0:
            ax.text(0.03, 0.97, dye, transform=ax.transAxes, ha="left", va="top", color="w", fontsize=7)
        if c == len(DYES) - 1:
            cb = plt.colorbar(im, ax=ax, pad=0.02)
            cb.locator = MaxNLocator(5)   # round ticks, not the 100 level boundaries
            cb.update_ticks()
            cb.set_label(label, fontsize=7)
plt.savefig(FIG_DIR / "Fig1c_error_metrics.svg", dpi=600, format="svg")
plt.show()

---
## 4 — Numbers at the cameras

Each camera of Fig. 1c at its nearest grid point (read noise and peak QE from `Camera_History.ods`).

In [ ]:
rows = []
for cam in camera_data.itertuples():
    j = int(np.argmin(np.abs(np.log(READ_NOISE) - np.log(cam.RMS))))
    i = int(np.argmin(np.abs(PEAK_QE - cam.PeakQY)))
    for dye in DYES:
        s = summary[(summary["dye"] == dye) & (summary["i_qe"] == i) & (summary["j_rn"] == j)].iloc[0]
        rows.append(dict(camera=cam.Model, dye=dye, read_noise=round(READ_NOISE[j], 2), peak_qe=round(PEAK_QE[i], 2),
                         yield_pct=100 * s["yield_"], rejected_pct=100 * s["frac_rejected"],
                         outside_3sigma_pct=100 * s["frac_outside_3sigma"],
                         pos_err_nm_median=s["pos_err_nm_median_accepted"], pos_err_nm_p95=s["pos_err_nm_p95_accepted"],
                         fp_err_median=s["fp_err_median_accepted"], fp_err_p95=s["fp_err_p95_accepted"],
                         sigma_err_nm_median=s["sigma_err_pos_nm_median_accepted"]))
at_cameras = pd.DataFrame(rows)
at_cameras.to_csv(FIG_DIR / "Fig1c_error_metrics_at_cameras.csv", index=False)
at_cameras.round(3)

In [ ]:
# Inside the 80% yield region, per dye: how large are the absolute errors?
inside = summary[summary["yield_"] >= YIELD_BENCHMARK]
inside.groupby("dye")[["pos_err_nm_median_accepted", "pos_err_nm_p95_accepted", "fp_err_median_accepted",
                       "fp_err_p95_accepted", "frac_rejected"]].agg(["median", "max"]).round(3)